# Section 4 · Notebook 1 — Supervisor–Worker Pattern (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

A market-research assistant:
- **News Agent** — recent news and developments (Tavily web search)
- **Financial Agent** — prices, history and fundamentals (Yahoo Finance via `yfinance`)
- **Sentiment Agent** — what investors are saying (Tavily web search)
- **Supervisor** — splits the question, delegates to the workers, and writes one recommendation

```
                    ┌──────────────┐
   question ──────► │  Supervisor  │ ──────► one combined answer
                    └──────┬───────┘
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
     News Agent     Financial Agent   Sentiment Agent
```

**You need** a free [Tavily](https://www.tavily.com/) API key. An [Alpha Vantage](https://www.alphavantage.co/support/#api-key) key is optional.

In [5]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

# --- FORCE ENVIRONMENT CLEAN UP ---
# This wipes out any hidden, cached environment variables causing the 404
os.environ.pop("GROQ_BASE_URL", None)
os.environ.pop("GROQ_LLM_MODEL", None)
os.environ.pop("GROQ_MODEL_ID", None)

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by deaulft; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)

print(f"Using {MODEL_ID} on Groq")
demo_agent = Agent(model=model)
response = demo_agent("Hello, Groq! Can you tell me a joke?")
print()
print("-" * 60)
print(response)

Using openai/gpt-oss-120b on Groq
Sure thing! Here’s a classic for you:

**Why don’t scientists trust atoms?**

Because they **make up** everything! 😄
------------------------------------------------------------
Sure thing! Here’s a classic for you:

**Why don’t scientists trust atoms?**

Because they **make up** everything! 😄



## Configure API keys

Set them in your shell before starting Jupyter, or put them in a `.env` file next to `GROQ_API_KEY` (recommended):

```bash
export TAVILY_API_KEY="tvly-..."
export ALPHA_VANTAGE_API_KEY="..."   # optional
```

…or uncomment the lines below. Never commit a real key to a notebook.

In [7]:
# os.environ["TAVILY_API_KEY"] = "tvly-..."
# os.environ["ALPHA_VANTAGE_API_KEY"] = "..."

if not os.getenv("TAVILY_API_KEY"):
    print("⚠️  TAVILY_API_KEY not set — the news and sentiment agents cannot search the web.")
else:
    print("✓ TAVILY_API_KEY is configured")

if not os.getenv("ALPHA_VANTAGE_API_KEY"):
    print("ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).")
else:
    print("✓ ALPHA_VANTAGE_API_KEY is configured")

✓ TAVILY_API_KEY is configured
✓ ALPHA_VANTAGE_API_KEY is configured


In [8]:
import yfinance as yf
from strands import Agent, ToolContext, tool
from strands_tools.tavily import tavily_search


# Groq's free tier rejects any single request over 8,000 tokens ("Request too large").
# A default Tavily search returns ~3,700 tokens, so a worker that searches twice overflows.
# The workers use this wrapper instead: the same Tavily search, top 3 results (~1,100 tokens),
# at most twice per task, because every result is resent with the worker's next request.
@tool(context=True)
async def web_search(query: str, tool_context: ToolContext, time_range: str = "month") -> dict | str:
    """Search the web with Tavily and return the top 3 results.

    Args:
        query: What to search for
        time_range: How recent the results should be: day, week, month or year
    """
    searches = tool_context.agent.state.get("searches") or 0  # as_tool() resets state for each task
    if searches >= 2:
        return "Search limit reached: answer with the results you already have."
    tool_context.agent.state.set("searches", searches + 1)
    return await tavily_search(query=query, max_results=3, time_range=time_range)

## Custom financial tools (Yahoo Finance)

In [9]:
def _num(value, money=False, pct=False):
    """Format numbers safely — Yahoo sometimes leaves fields empty."""
    if not isinstance(value, (int, float)):
        return "N/A"
    if pct:
        return f"{value * 100:.2f}%"
    return f"${value:,.2f}" if money else f"{value:,.2f}"


@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price and key metrics using Yahoo Finance.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA', 'MSFT')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Stock: {symbol}\n"
            f"Current Price: {_num(info.get('currentPrice') or info.get('regularMarketPrice'), money=True)}\n"
            f"Market Cap: {_num(info.get('marketCap'), money=True)}\n"
            f"P/E Ratio: {_num(info.get('trailingPE'))}\n"
            f"52 Week High: {_num(info.get('fiftyTwoWeekHigh'), money=True)}\n"
            f"52 Week Low: {_num(info.get('fiftyTwoWeekLow'), money=True)}\n"
            f"Average Volume: {_num(info.get('averageVolume'))}\n"
            f"Sector: {info.get('sector', 'N/A')}\n"
            f"Industry: {info.get('industry', 'N/A')}"
        )
    except Exception as e:
        return f"Error fetching stock data for {symbol}: {e}"


@tool
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """Get historical stock price data.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
        period: One of 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    """
    try:
        hist = yf.Ticker(symbol).history(period=period)
        if hist.empty:
            return f"No historical data available for {symbol}"
        start_price = hist["Close"].iloc[0]
        end_price = hist["Close"].iloc[-1]
        change_pct = (end_price - start_price) / start_price * 100
        return (
            f"Historical Data for {symbol} ({period}):\n"
            f"Starting Price: ${start_price:.2f}\n"
            f"Ending Price: ${end_price:.2f}\n"
            f"Change: {change_pct:+.2f}%\n"
            f"Highest: ${hist['High'].max():.2f}\n"
            f"Lowest: ${hist['Low'].min():.2f}\n"
            f"Average Volume: {hist['Volume'].mean():,.0f}\n\n"
            f"Recent 5-day trend:\n{hist[['Close', 'Volume']].tail().to_string()}"
        )
    except Exception as e:
        return f"Error fetching historical data for {symbol}: {e}"


@tool
def get_company_financials(symbol: str) -> str:
    """Get company financial metrics: revenue, margins, growth and leverage.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Financial Metrics for {symbol}:\n"
            f"Revenue: {_num(info.get('totalRevenue'), money=True)}\n"
            f"Revenue Growth: {_num(info.get('revenueGrowth'), pct=True)}\n"
            f"Gross Profit: {_num(info.get('grossProfits'), money=True)}\n"
            f"Profit Margin: {_num(info.get('profitMargins'), pct=True)}\n"
            f"Operating Margin: {_num(info.get('operatingMargins'), pct=True)}\n"
            f"EBITDA: {_num(info.get('ebitda'), money=True)}\n"
            f"Earnings Growth: {_num(info.get('earningsGrowth'), pct=True)}\n"
            f"Return on Equity: {_num(info.get('returnOnEquity'), pct=True)}\n"
            f"Debt to Equity: {_num(info.get('debtToEquity'))}\n"
            f"Free Cash Flow: {_num(info.get('freeCashflow'), money=True)}"
        )
    except Exception as e:
        return f"Error fetching financial data for {symbol}: {e}"


print(get_stock_price(symbol="AAPL"))  # tools are still plain functions you can call

Stock: AAPL
Current Price: $347.73
Market Cap: $5,074,761,285,632.00
P/E Ratio: 39.88
52 Week High: $351.73
52 Week Low: $243.42
Average Volume: 43,991,589.00
Sector: Technology
Industry: Consumer Electronics


## Optional: Alpha Vantage tool

Use this instead of (or alongside) yfinance if you have an Alpha Vantage key. Other providers worth knowing: Polygon.io, Finnhub, Financial Modeling Prep.

In [10]:
from alpha_vantage.timeseries import TimeSeries


@tool
def get_alpha_vantage_quote(symbol: str) -> str:
    """Get the latest stock quote from the Alpha Vantage API.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL')
    """
    key = os.getenv("ALPHA_VANTAGE_API_KEY")
    if not key:
        return "ALPHA_VANTAGE_API_KEY is not set."
    ts = TimeSeries(key=key, output_format="pandas")
    data, _ = ts.get_quote_endpoint(symbol=symbol)
    return data.to_string()


financial_tools = [get_stock_price, get_stock_history, get_company_financials, web_search]
if os.getenv("ALPHA_VANTAGE_API_KEY"):
    financial_tools.append(get_alpha_vantage_quote)

## Worker agents

In [11]:
news_agent = Agent(
    name="news_agent",
    description="Researches recent news and developments about companies and industries.",
    model=model,
    system_prompt="""You are a news research specialist. Search for recent news and developments about companies and industries.

When searching:
- Focus on recent news (use time_range='week' or 'month')
- Prefer reliable sources such as major news outlets
- Summarize key developments and trends
- Include specific facts, dates, and figures when available
- Make at most two web_search calls: Groq's free tier caps each request at 8,000 tokens""",
    tools=[web_search],
    callback_handler=None,
)

financial_agent = Agent(
    name="financial_agent",
    description="Analyzes stock prices, price history, and company financial metrics.",
    model=model,
    system_prompt="""You are a financial analysis specialist. Analyze financial metrics, stock performance, and company valuations.

When analyzing:
- Use get_stock_price for current prices and key metrics
- Use get_stock_history for price trends
- Use get_company_financials for revenue, earnings, and growth
- Compare companies within the same sector and give specific numbers
- Look at no more than three companies, and use web_search at most once""",
    tools=financial_tools,
    callback_handler=None,
)

sentiment_agent = Agent(
    name="sentiment_agent",
    description="Analyzes market and investor sentiment from forums, social media, and commentary.",
    model=model,
    system_prompt="""You are a market sentiment analyst. Analyze market sentiment from social media, forums, and investor discussions.

When analyzing sentiment:
- Look for both bullish and bearish opinions
- Identify common concerns and excitement points
- Distinguish retail from institutional sentiment
- Summarize the overall market mood
- Make at most two web_search calls: Groq's free tier caps each request at 8,000 tokens""",
    tools=[web_search],
    callback_handler=None,
)
print("✓ Worker agents created")

✓ Worker agents created


## Supervisor agent

`agent.as_tool()` wraps a worker so the supervisor can call it like any other tool.

> **Groq note:** `gpt-oss` tends to answer a broad question like "Should I invest in EV companies?" by asking the user to narrow it down, instead of delegating. The last line of the system prompt tells it to pick sensible defaults and delegate straight away. (Passing the agents directly in `tools=[...]` does the same thing.)

In [12]:
supervisor = Agent(
    name="research_supervisor",
    model=model,
    system_prompt="""You are a market research coordinator. When asked about investment opportunities:

1. Work out what information is needed
2. Delegate to the specialist agents:
   - news_agent: recent developments and industry news
   - financial_agent: financial metrics and performance data
   - sentiment_agent: market sentiment and investor opinions
3. Synthesize their responses into one balanced recommendation

This is educational analysis, not financial advice — say so briefly at the end.
Do not ask the user clarifying questions: pick sensible defaults (for example, the sector's leading companies) and delegate straight away.""",
    tools=[news_agent.as_tool(), financial_agent.as_tool(), sentiment_agent.as_tool()],
)
print("✓ Supervisor created")

✓ Supervisor created


## Run the multi-agent system

In [13]:
question = "Should I invest in electric vehicle companies?"
print(f"Question: {question}\n")

response = supervisor(question)

Question: Should I invest in electric vehicle companies?


Tool #1: news_agent


DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Tesla BYD NIO Rivian Lucid news week                                                                     │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Tesla GigaBerlin Sets Weekly Record as Plant Runs 24/5 and Eyes New Hall                                    │
│ URL:                                                                                                            │
│ https://eletric-vehicles.com/tesla/tesla-gigaberlin-sets-weekly-record-as-plant-runs-24-5-and-eyes-new-hall     │
│ Score: 0.64986813                                                                                               │
│ Content: Trending searches                                                                                      │
│                                                                                                                 │
│  Nio                                                                                                            │
│  Tesla                                                                                                          │
│  Lucid                                                                                                          │
│  Rivian                                                                                                         │
│  XPeng                                                                                                          │
│  Onvo                                                                                                           │
│  BYD                                                                                                            │
│  Polestar                                                                                                       │
│                                                                                                                 │
│ Latest results                                                                                                  │
│                                                                                                                 │
│  TeslaTesla GigaBerlin Sets Weekly Record as Plant Runs 24/5 and Eyes New HallJust now                          │
│  BYDBYD Cars Banned From US Kadena Air Base in Japan Over Security Risk20h ago                                  │
│  TeslaTesla Cybercab Turn-Signal Hurdle Eased in NHTSA Lighting Proposal20h ago                                 │
│  CanadaChery Says Canada Is Worth a Factory Even Without Access to the US23h ago                                │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Are EVs Profitable? Why Most EV Makers Are Losing Money                                                     │
│ URL: https://taasmaster.com/p/new-powertrain-same-car-business                                                  │
│ Score: 0.55709904                                                                                               │
│ Content: Automaker results                                                                                      │
│                                                                                                                 │
│ CnEVPost, BYD H1 2026 earnings, August 28, 2026       

DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Tesla earnings week Reuters                                                                              │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Tesla has a surprise for investors: a big cut in investment | Reuters                                       │
│ URL: https://www.reuters.com/video/watch/idRW216822102026RP1                                                    │
│ Score: 0.49670476                                                                                               │
│ Content: # Reuters Video                                                                                        │
│                                                                                                                 │
│ ## Up next in Autos and Transportation (25 videos)                                                              │
│                                                                                                                 │
│  Tesla has a surprise for investors: a big cut in investment                                                    │
│  Tech Weekly: Tesla's Roadster 'takes flight', Japan heads for Phobos                                           │
│                                                                                                                 │
│   02:30                                                                                                         │
│  Why VW can't seem to catch a break in China, even with local help                                              │
│                                                                                                                 │
│   02:21                                                                                                         │
│  We have liftoff: Tesla's new Roadster can fly, sort of                                                         │
│                                                                                                                 │
│   02:39                                                                                                         │
│  How China aims to beat European carmakers on their own roads                                                   │
│                                                                                                                 │
│   03:53                                                                                                         │
│  Back to the future: the electric Citroen 2CV [...] ## Up next in Autos and Transportation (25 videos)          │
│                                                                                                                 │
│  Tesla has a surprise for investors: a big cut in investment                                                    │
│  Tech Weekly: Tesla's Roadster 'takes flight', Japan heads for Phobos                                           │
│                                                                                                                 │
│   02:30                                                                                                         │
│  Why VW can't seem to catch a break in China, even with local help                                              │
│                                                                                                                 │
│   02:21                                               

...

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Tesla BYD NIO Rivian Lucid investor sentiment past month social media Reddit Twitter StockTwits          │
│ discussion profitability valuation new product launches                                                         │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Q3 2026 EV Sales Roundup: BYD, Tesla, Rivian & More                                                         │
│ URL: https://motorwatt.com/ev-blog/trends/q3-2026-ev-sales-roundup                                              │
│ Score: 0.7055624                                                                                                │
│ Content: | NIO | 109,178 | +25.4% | Three brands (NIO, ONVO, Firefly); the new ES8 passed 150,000 units |       │
│ | Rivian | 19,248 | +45.8% | First full quarter of R2 deliveries; company record |                              │
│ | Lucid | 3,806 | n/a | Inventory clearance; production cut to a single shift | [...] 762,478                   │
│                                                                                                                 │
│ Tesla                                                                                                           │
│                                                                                                                 │
│ 486,532                                                                                                         │
│                                                                                                                 │
│ Leapmotor                                                                                                       │
│                                                                                                                 │
│ 310,052                                                                                                         │
│                                                                                                                 │
│ XPeng                                                                                                           │
│                                                                                                                 │
│ 118,000+                                                                                                        │
│                                                                                                                 │
│ NIO                                                                                                             │
│                                                                                                                 │
│ 109,178                                                                                                         │
│                                                                                                                 │
│ Rivian                                                                                                          │
│                                                                                                                 │
│ 19,248                                                                                                          │
│                                                                                                                 │
│ Lucid                                                 

DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Tesla Reddit sentiment October 2026 profitability valuation new model                                    │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Tesla dominates Reddit talk with 785 comments, but sentiment is souring | 24/7 Wall St.                     │
│ URL:                                                                                                            │
│ https://247wallst.com/cards/tesla-leads-the-board-with-785-comments-though-the-crowd-is-tsla-reddit-radar-01m5h │
│ x6vr77xw077ss6hnhdaea                                                                                           │
│ Score: 0.834852                                                                                                 │
│ Content: # Tesla dominates Reddit talk with 785 comments, but sentiment is souring                              │
│                                                                                                                 │
│ As seen on the 24/7 Wall St. homepage on October 22, 2026.                                                      │
│                                                                                                                 │
│ ▲ASTSheating up fast                                                                                            │
│                                                                                                                 │
│ ▼TSLAsouring fast                                                                                               │
│                                                                                                                 │
│ Tesla leads the board with 785 comments, though the crowd is split and souring after earnings. AST SpaceMobile  │
│ is gaining traction, with Intel and Nvidia also drawing bullish talk. [...] Tesla is the most-discussed stock   │
│ on Reddit's investing communities today with 785 comments, and the mood is trending negative after the          │
│ company's latest earnings report. It is the clearest faller in today's sentiment snapshot.                      │
│                                                                                                                 │
│ Meta Platforms is the second most-talked-about name with 571 comments, and its community is also divided.       │
│ Upvotes are outpacing replies, pointing to a quieter but more engaged audience.                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Tesla's Profits Are Falling, but Our AI Score Is Turning Bullish. Is TSLA a Buy? - AltIndex                 │
│ URL: https://altindex.com/news/tesla-profit-down-ai-score-up                                                    │
│ Score: 0.6991933                                                                                                │
│ Content: Tesla's latest earnings show a company struggling to grow profits while spending heavily on its next   │
│ generation of products. Yet our AI Score has been climbing for months, supported by stronger hiring activity,   │
│ improving employee confidence and recovering retail sentiment.                                                  │
│                                                       

OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
exception=<ModelThrottledException> | event loop cycle failed


EventLoopException: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kyqc6cn3ftes9e1zwdam02r7` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 10714, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Try different questions

> **Groq note:** each question costs about a dozen model calls (roughly 30k tokens), and Groq's free tier allows 8,000 tokens a minute and 200k a day per model. So the loop asks one more question (uncomment the second if you have budget to spare), and it clears the supervisor's conversation first: the previous question's worker reports would otherwise push a single request past Groq's 8,000-token limit. Strands waits and retries at the per-minute limit, so a question can take a few minutes.

In [ ]:
questions = [
    "What's the outlook for renewable energy stocks?",
    # "Is the semiconductor industry a good investment right now?",  # see the Groq note above
]

for q in questions:
    print(f"\n{'=' * 80}\nQuestion: {q}\n{'=' * 80}")
    supervisor.messages = []  # start each question fresh (see the Groq note above)
    supervisor(q)

## Key takeaways

1. **Specialisation** — each worker has one job and the right tools for it
2. **Coordination** — the supervisor decides who to ask and combines the answers
3. **Modularity** — add a new worker without touching the others

**Make it production-ready:** retries and caching for API calls, rate limits, async execution, and tracing (Section 7).